In [ ]:
import numpy as np
import pandas as pd
import os
import scipy
from scipy.stats import sem

## Configuration

In [ ]:
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"] # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
domain = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
domain_type = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
model_name = "CLIP_ViT_Vision" # DeiT | CLIP_ViT_Vision | Google_ViT
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
indices = [i for i in range(12)]
size = [i for i in range(1,6)]

## Loading Data

In [ ]:
def sort(name):
    split = name.split("Results_")
    return int(split[1][0])

In [ ]:
for k in range(len(dataset_name)):
    results_path = f"../Data/Increment_Training/{dataset_name[k]}/{domain}/Entire_Transformation_Matrix_W"

    standard = []

    try:
        for filename in os.listdir(f"{results_path}/Standard"):
            if filename in [".DS_Store"]:
                continue
            file_path = os.path.join(f"{results_path}/Standard", filename)
            if os.path.isfile(file_path):
                standard.append(file_path)
    except FileNotFoundError:
        print(f"Error: The Folder '{results_path}/Standard' was not found.")
    except Exception as e:
        print(f"An error occured: {e}")

    standard = sorted(standard, key=lambda df: sort(df))
    standard = [pd.read_json(i) for i in standard]

    # Fine_Tuned_Classifier
    fine_tuned_classifier = []

    try:
        for filename in os.listdir(f"{results_path}/Fine_Tuned_Classifier"):
            if filename in [".DS_Store"]:
                continue
            file_path = os.path.join(f"{results_path}/Fine_Tuned_Classifier", filename)
            if os.path.isfile(file_path):
                fine_tuned_classifier.append(file_path)
    except FileNotFoundError:
        print(f"Error: The Folder '{results_path}/Standard' was not found.")
    except Exception as e:
        print(f"An error occured: {e}")

    fine_tuned_classifier = sorted(fine_tuned_classifier, key=lambda df: sort(df))
    fine_tuned_classifier = [pd.read_json(i) for i in fine_tuned_classifier]

    # Linear_Probe
    linear_probe = []

    try:
        for filename in os.listdir(f"{results_path}/Linear_Probe"):
            if filename in [".DS_Store"]:
                continue
            file_path = os.path.join(f"{results_path}/Linear_Probe", filename)
            if os.path.isfile(file_path):
                linear_probe.append(file_path)
    except FileNotFoundError:
        print(f"Error: The Folder '{results_path}/Linear_Probe' was not found.")
    except Exception as e:
        print(f"An error occured: {e}")

    linear_probe = sorted(linear_probe, key=lambda df: sort(df))
    linear_probe = [pd.read_json(i) for i in linear_probe]

    standard_acc = {i: [] for i in indices}
    fine_tuned_acc = {i: [] for i in indices}
    linear_probe_acc = {i: [] for i in indices}

    for i in indices:
        for j in range(5):
            standard_acc[i].append(standard[j]["Classification_Accuracy"][i])
            fine_tuned_acc[i].append(fine_tuned_classifier[j]["Classification_Accuracy"][i])
            linear_probe_acc[i].append(linear_probe[j]["Classification_Accuracy"][i])
    
    standard_acc_mean = [np.mean(i) for i in standard_acc.values()]
    fine_tuned_acc_mean = [np.mean(i) for i in fine_tuned_acc.values()]
    linear_probe_acc_mean = [np.mean(i) for i in linear_probe_acc.values()]

    standard_index = standard_acc_mean.index(max(standard_acc_mean))
    fine_tuned_index = fine_tuned_acc_mean.index(max(fine_tuned_acc_mean))
    linear_probe_index = linear_probe_acc_mean.index(max(linear_probe_acc_mean))

    standard_vals = standard_acc[standard_index]
    fine_tuned_classifier_vals = fine_tuned_acc[fine_tuned_index]
    linear_probe_classifier_vals = linear_probe_acc[linear_probe_index]

    print(f"{dataset_name[k]}")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(standard_vals)-1, loc=np.mean(standard_vals), scale=sem(standard_vals))
    print(f"Task Matrix Average: {np.mean(standard_vals)}, Error: +- {ci_high-np.mean(standard_vals)}, 95% Confidence Interval: {(ci_low, ci_high)}, Index: {standard_index}")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(fine_tuned_classifier_vals)-1, loc=np.mean(fine_tuned_classifier_vals), scale=sem(fine_tuned_classifier_vals))
    print(f"Base w/Fine-Tuned Classifier Average: {np.mean(fine_tuned_classifier_vals)}, Error: +- {ci_high-np.mean(fine_tuned_classifier_vals)}, 95% Confidence Interval: {(ci_low, ci_high)}, Index: {fine_tuned_index}")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(linear_probe_classifier_vals)-1, loc=np.mean(linear_probe_classifier_vals), scale=sem(linear_probe_classifier_vals))
    print(f"Linear Probe Average: {np.mean(linear_probe_classifier_vals)}, Error: +- {ci_high-np.mean(linear_probe_classifier_vals)}, 95% Confidence Interval: {(ci_low, ci_high)}, Index: {linear_probe_index}")
    print()

In [ ]:
DTD = []
EuroSAT = []
GTSRB = []
MNIST = []
RESISC45 = []
Stanford_Cars = []
SUN397 = []
SVHN = []